In [ ]:
""" Configuration cell — thesis figures (see paper_plots.ipynb for the paper ones)"""

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

import morethemes as mt
mt.set_theme("minimal")

VERIF_DIR = Path("runs/verification_case")
CONV_DIR = Path("runs/convergence_study_2")
AV_DIR = Path("runs/analytical_validity")
FIG_DIR = Path("thesis_fig")                  # copy into masters_thesis/fig once happy with them
FIG_DIR.mkdir(exist_ok=True, parents=True)

HOURS_TO_SEC = 3600
SEC_TO_HOURS = 1 / HOURS_TO_SEC
M_TO_CM = 1e2

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["STIXGeneral", "Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.7,
    "lines.linewidth": 1.3,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": False,
    "ytick.right": False,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,
    "xtick.minor.size": 2,
    "ytick.minor.size": 2,
    "xtick.major.width": 0.7,
    "ytick.major.width": 0.7,
    "xtick.minor.width": 0.5,
    "ytick.minor.width": 0.5,
    "legend.frameon": False,
    "savefig.bbox": "tight",
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

In [ ]:
# %% ===================== HELPERS =====================
def _read_units(path):
    """Read the leading '# units: <unit>' comment line, if present."""
    with open(path) as f:
        first = f.readline().strip()
    m = re.match(r"#\s*units:\s*(.*)", first)
    return m.group(1) if m else ""


def load_export(path: Path):
    """Load an export, inferring its kind from the CSV column layout.
        profile : col 0 'x_metres', then one column per time step 't=<s>s'
        static  : columns 'x_metres', 'value'
        series  : columns 't_seconds', 'value'
    """
    unit = _read_units(path)
    df = pd.read_csv(path, comment="#")
    cols = list(df.columns)
    if cols[0] == "x_metres" and all(c.startswith("t=") for c in cols[1:]):
        times = np.array([float(re.search(r"t=([\d.eE+-]+)s", c).group(1)) for c in cols[1:]])
        return dict(kind="profile", unit=unit, x=df["x_metres"].to_numpy(),
                    times=times, data=df[cols[1:]].to_numpy().T)
    elif cols == ["x_metres", "value"]:
        return dict(kind="static", unit=unit, x=df["x_metres"].to_numpy(),
                    data=df["value"].to_numpy())
    elif cols == ["t_seconds", "value"]:
        return dict(kind="series", unit=unit, times=df["t_seconds"].to_numpy(),
                    data=df["value"].to_numpy())
    raise ValueError(f"Unrecognized export layout in {path}: columns={cols}")


def select_nearest(times, targets):
    """For each target time, return the index of the closest available time."""
    return [int(np.argmin(np.abs(times - t))) for t in targets]


def richardson(f, r):
    """Observed order p, extrapolate and GCI from the finest 3 of a coarse->fine
    series (NaN when the triplet is non-monotone). Roache/Celik GCI convention:
    f1 = finest grid, f2 = next-coarsest, f3 = coarsest of the three."""
    f3, f2, f1 = f[-3:]
    eps21, eps32 = f2 - f1, f3 - f2
    if eps32 == 0 or eps21 == 0 or np.sign(eps21) != np.sign(eps32):
        return np.nan, f1, np.nan
    p = np.log(abs(eps32) / abs(eps21)) / np.log(r)
    f_exact = f1 + (f1 - f2) / (r**p - 1)
    gci = 1.25 * abs((f1 - f2) / f1) / (r**p - 1)
    return p, f_exact, gci


def ref_slope(ax, xdata, slope, xref, eref, label, color="0.45"):
    """Dashed reference guide of order `slope` through (xref, eref)."""
    x = np.array([xdata.min(), xdata.max()])
    y = eref * (x / xref) ** slope
    ax.plot(x, y, ls="--", lw=0.9, color=color, zorder=0)
    ax.text(x[0] * 1.15, y[0] * 1.4, label, color=color, fontsize=7, va="bottom", ha="left")

In [ ]:
# %% ===================== CELL V0 : load the verification case =====================
""" Verification against the analytical solution — data from
    paper.generate_paper_data.verification_case() """
import sys
sys.path.insert(0, ".")
from generate_paper_data import _make_scaled_input
from sparging import get_sim_input_LIBRA_Pi, ureg
from scipy.integrate import quad

VARIANT_LABEL = {"spp": r"SPP, $\langle\Pi\rangle = %.3g$", "ppl": r"PPL, $\langle\Pi\rangle = %.3g$"}
PANEL = {"spp": "(a)", "ppl": "(b)"}
SIM_KW = dict(lw=2.6, alpha=0.45, solid_capstyle="round")          # thick, faded: the model
ANA_KW = dict(color="0.15", lw=1.0, ls=(0, (3, 2)), zorder=5)      # thin, crisp: the analytical
VARIANT_COLOR = {"spp": "#4477AA", "ppl": "#EE7733"}

VERIF = {}
for variant in ("spp", "ppl"):
    d = VERIF_DIR / variant
    ard = pd.read_csv(d / "ard_inputs.csv").set_index("key")
    meta = json.load(open(d / "metadata.json"))
    inp = _make_scaled_input(
        get_sim_input_LIBRA_Pi().height * meta["height_scale"], "K_s", meta["k_s_scale"]
    )
    VERIF[variant] = dict(
        ard=ard, meta=meta, inp=inp,
        tau_s=inp.get_tau().to("s").magnitude,
        Pi=float(ard.loc["Pi_ave", "value"]),
        H=float(ard.loc["H", "value"]),
        A=float(ard.loc["A", "value"]),
        K_s=float(ard.loc["K_s", "value"]),
        dt_s=float(ard.loc["dt", "value"]),
        P_T2=load_export(d / "P_T2.csv"),
        inventory=load_export(d / "n_T2_salt.csv"),
    )
    print(f"{variant}: Pi={VERIF[variant]['Pi']:.3g}, G_P={float(ard.loc['G_P','value']):.3g}, "
          f"G_mix={float(ard.loc['G_mix','value']):.2g}, tau={VERIF[variant]['tau_s']/3600:.2f} h, "
          f"n_cells={int(ard.loc['n_cells','value'])}, n_steps={int(ard.loc['n_steps','value'])}")


def pi_star(inp, z_m):
    """Cumulated partial pressure number seen by a bubble from the sparger up to z:
    Pi*(z) = (1/H) int_0^z Pi(s) ds, so that Pi*(H) = <Pi>."""
    H = inp.height.to("m").magnitude
    f = lambda s: inp.get_Pi(s * ureg.m).magnitude
    return np.array([quad(f, 0, zi)[0] / H for zi in np.atleast_1d(z_m)])

In [ ]:
# %% ===================== CELL V1 : gas partial pressure profile vs analytical =====================
# analytical: P_T2(z) = <c_T2>^l / K_s * (1 - exp(-Pi*(z))), with <c_T2>^l taken from the
# simulated inventory at the same instant (the liquid is uniform: G_mix ~ 1e-6).
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.9))

for ax, variant in zip(axes, ("spp", "ppl")):
    v = VERIF[variant]
    e, inv = v["P_T2"], v["inventory"]
    idx = select_nearest(e["times"], np.array([0.2, 1.0, 2.0]) * v["tau_s"])
    colors = plt.cm.viridis(np.linspace(0.05, 0.75, len(idx)))
    zs = np.linspace(0, v["H"], 200)
    saturation = 1 - np.exp(-pi_star(v["inp"], zs))

    err = []
    for c, i in zip(colors, idx):
        c_mean = np.interp(e["times"][i], inv["times"], inv["data"]) / (v["A"] * v["H"])
        ana = c_mean / v["K_s"] * saturation
        ax.plot(e["x"] * M_TO_CM, e["data"][i], color=c, **SIM_KW,
                label=fr"$t = {e['times'][i] / HOURS_TO_SEC:.0f}\,$h")
        ax.plot(zs * M_TO_CM, ana, **ANA_KW)
        err.append(np.max(np.abs(np.interp(e["x"], zs, ana) - e["data"][i])) / np.max(ana))

    ax.set_xlabel(r"$z\ \mathrm{[cm]}$")
    ax.set_title(f"{PANEL[variant]} " + VARIANT_LABEL[variant] % v["Pi"], fontsize=9)
    ax.set_xlim(0, v["H"] * M_TO_CM)
    ax.legend(loc="upper left", fontsize=7)
    print(f"{variant}: max |P_T2 - analytical| / max(P_T2) = {max(err):.2e}")

axes[0].set_ylabel(r"$P_{T_2}\ \mathrm{[Pa]}$")
axes[0].plot([], [], color="0.4", **SIM_KW, label="1D ARD model")
axes[0].plot([], [], **ANA_KW, label="analytical")
handles = axes[0].get_legend_handles_labels()
axes[1].legend(handles[0][-2:], handles[1][-2:], loc="lower right", fontsize=7)

fig.tight_layout()
fig.savefig(FIG_DIR / "verification_P_T2_profile.pdf")

In [ ]:
# %% ===================== CELL V2 : inventory decay vs analytical =====================
# n_T2(t) = n_T2(0) exp(-t/tau), plotted against absolute time so the two regimes keep their
# own decay rate (normalising by tau would collapse them onto the same curve).
fig, (ax, axr) = plt.subplots(2, 1, figsize=(3.5, 3.6), sharex=True,
                              gridspec_kw={"height_ratios": [2.2, 1]})

for variant in ("spp", "ppl"):
    v = VERIF[variant]
    t, n = v["inventory"]["times"], v["inventory"]["data"]
    ana = n[0] * np.exp(-t / v["tau_s"])
    ana_dt = n[0] * np.exp(-t / (v["tau_s"] + v["dt_s"] / 2))
    color = VARIANT_COLOR[variant]

    ax.plot(t / HOURS_TO_SEC, n, color=color, **SIM_KW,
            label=VARIANT_LABEL[variant] % v["Pi"])
    ax.plot(t / HOURS_TO_SEC, ana, **ANA_KW)
    axr.plot(t / HOURS_TO_SEC, (n - ana) / ana * 100, color=color, lw=1.0, ls="-")
    axr.plot(t / HOURS_TO_SEC, (n - ana_dt) / ana_dt * 100, color=color, lw=1.0, ls="--")
    print(f"{variant}: max deviation vs tau = {np.max(np.abs(n - ana) / ana):.2e}, "
          f"vs tau + dt/2 = {np.max(np.abs(n - ana_dt) / ana_dt):.2e}")

ax.set_yscale("log")
ax.set_ylabel(r"$n_{T_2}\ \mathrm{[mol]}$")
ax.plot([], [], **ANA_KW, label="analytical")
ax.legend(loc="lower left", fontsize=7)
axr.axhline(0, color="0.6", lw=0.6)
axr.set_xlabel(r"$t\ \mathrm{[h]}$")
axr.set_ylabel("residual [%]", fontsize=8)
axr.plot([], [], color="0.3", lw=1.0, ls="-", label=r"vs $\tau$")
axr.plot([], [], color="0.3", lw=1.0, ls="--", label=r"vs $\tau + \Delta t/2$")
axr.legend(loc="lower left", fontsize=6, ncol=2)

fig.tight_layout()
fig.savefig(FIG_DIR / "verification_inventory.pdf")

In [ ]:
# %% ===================== CELL C0 : load the convergence study =====================
""" Mesh / time-step convergence on the nominal LIBRA-Pi input """
_cs = json.load(open(CONV_DIR / "convergence_data.json"))
CS_META = _cs["metadata"]
R_REF = CS_META["refinement_ratio"]
TAU_ANA = CS_META["tau_s"] * SEC_TO_HOURS


def cs_sweep(axis):
    """Return (dt[s], dx[m], tau_fitted[h]) arrays, ordered coarse -> fine."""
    recs = _cs[f"{axis}_sweep"]
    return (np.array([x["dt_s"] for x in recs]),
            np.array([x["dx_m"] for x in recs]),
            np.array([x["tau_fitted_s"] for x in recs]) * SEC_TO_HOURS)


P_T, TAU_INF_T, GCI_T = richardson(cs_sweep("dt")[2], R_REF)
P_X, TAU_INF_X, GCI_X = richardson(cs_sweep("dx")[2], R_REF)
print(f"Pi={CS_META['Pi']:.3f}, Bo={CS_META['Bo']:.1f}, analytical tau={TAU_ANA:.4f} h")
print(f"time  : p={P_T:.3f}, tau_inf={TAU_INF_T:.5f} h, GCI={GCI_T*100:.3f} %")
print(f"space : p={P_X:.3f}, tau_inf={TAU_INF_X:.5f} h, GCI={GCI_X*100:.2e} %")

In [ ]:
# %% ===================== CELL C1 : log-log refinement =====================
# discretisation error of tau_fitted against the Richardson extrapolate of each sweep
fig, (axt, axx) = plt.subplots(1, 2, figsize=(7.0, 2.9))

dt, _, tau_t = cs_sweep("dt")
_, dx, tau_x = cs_sweep("dx")
e_t = np.abs(tau_t - TAU_INF_T) / TAU_INF_T
e_x = np.abs(tau_x - TAU_INF_X) / TAU_INF_X

axt.loglog(dt / (TAU_ANA * HOURS_TO_SEC), e_t, "o-", color="#4477AA", ms=4, mfc="white")
ref_slope(axt, dt / (TAU_ANA * HOURS_TO_SEC), 1.0, dt[-1] / (TAU_ANA * HOURS_TO_SEC),
          e_t[-1], r"order 1")
axt.set_xlabel(r"$\Delta t / \tau$")
axt.set_ylabel(r"$|\tau_\mathrm{fit} - \tau_\infty| / \tau_\infty$")
axt.set_title("time step", fontsize=9)

axx.loglog(dx, e_x, "s-", color="#228833", ms=4, mfc="white")
ref_slope(axx, dx, 2.0, dx[-1], e_x[-1], r"order 2")
axx.set_xlabel(r"$\Delta x\ \mathrm{[m]}$")
axx.set_title("mesh", fontsize=9)

for ax, lab in zip((axt, axx), ("(a)", "(b)")):
    ax.text(0.03, 0.95, lab, transform=ax.transAxes, fontweight="bold", va="top")

fig.tight_layout()
fig.savefig(FIG_DIR / "convergence_refinement.pdf")

In [ ]:
# %% ===================== CELL C2 : convergence summary table =====================
# Roache/Celik verification metrics + the backward Euler prediction tau_num = tau + dt/2,
# i.e. a relative error dt/(2 tau) that the ratio below should reproduce.
dt, _, tau_t = cs_sweep("dt")
_, dx, tau_x = cs_sweep("dx")
ratio = (tau_t[-1] / TAU_INF_T - 1) / (dt[-1] / (TAU_INF_T * HOURS_TO_SEC))

conv_df = pd.DataFrame([
    {"quantity": "observed order of convergence $p$",
     "time step": f"{P_T:.3f}", "mesh": f"{P_X:.3f}"},
    {"quantity": r"Richardson extrapolate $\tau_\infty$ [h]",
     "time step": f"{TAU_INF_T:.4f}", "mesh": f"{TAU_INF_X:.4f}"},
    {"quantity": r"GCI on the finest discretisation [\%]",
     "time step": f"{GCI_T * 100:.3f}", "mesh": f"{GCI_X * 100:.1e}"},
    {"quantity": "finest discretisation",
     "time step": f"$\\Delta t / \\tau = {dt[-1] / (TAU_ANA * HOURS_TO_SEC):.1e}$",
     "mesh": f"$\\Delta x = {dx[-1]:.1e}$ m"},
    {"quantity": r"error on $\tau_\infty$ there [\%]",
     "time step": f"{(tau_t[-1] / TAU_INF_T - 1) * 100:.3f}",
     "mesh": f"{(tau_x[-1] / TAU_INF_X - 1) * 100:.1e}"},
]).set_index("quantity")
print(conv_df.to_string())
print(f"\nanalytical tau = {TAU_ANA:.4f} h -> tau_inf/tau_analytical - 1 = "
      f"{TAU_INF_T / TAU_ANA - 1:+.2e}")
print(f"measured [tau_fit(dt)/tau_inf - 1] / (dt/tau) = {ratio:.3f}  (backward Euler predicts 0.5)")

# LaTeX fragment for \input in the thesis (booktabs)
_ltx = conv_df.to_latex(
    escape=False, column_format="lrr",
    caption=(f"Grid convergence of the fitted decay time $\\tau_\\mathrm{{fit}}$ on the nominal "
             f"LIBRA-Pi input ($\\langle\\Pi\\rangle = {CS_META['Pi']:.3f}$), refinement ratio "
             f"$r = {R_REF:g}$, Roache safety factor $\\mathrm{{F_s}} = 1.25$."),
    label="tab:convergence_gci", position="htbp",
)
(FIG_DIR / "convergence_table.tex").write_text(_ltx)
print("wrote", FIG_DIR / "convergence_table.tex")

In [ ]:
# %% ===================== CELL A0 : load the analytical validity studies =====================
""" Validity of the analytical extraction time against the 1D ARD model, over a sampled space,
    correlated with the groups of the three assumptions (Pi, G_mix, G_P).
    Two designs are loaded and every figure below is produced for both:
      - analytical_validity  : h_l/K_s multiplier and tank height H  -> figures in thesis_fig/
      - analytical_validity2 : K_s, P_top and E_l, one per group     -> figures in its run folder """
AV2_DIR = Path("runs/analytical_validity2")

BASE_COLOR = "#4477AA"
SPP_COLOR = "0.6"
FLAG_EDGE_COLOR = "#CC3311"
BASE_EDGE_COLOR = "0.25"


def load_validity(run_dir: Path, csv_name: str, fig_dir: Path) -> dict:
    """Load one validity study and derive the discretisation-free error columns."""
    meta = json.load(open(run_dir / "metadata.json"))
    df = pd.read_csv(run_dir / csv_name)
    # every figure is produced twice: from the raw fitted tau, and with the backward Euler
    # bias dt/2 removed from it, which leaves the model error alone (suffix "_nodisc")
    df["tau_fit_nodisc_s"] = df.tau_fitted_s - df.dt_s / 2
    for tag, pred in (("pred", "tau_pred_s"), ("ave", "tau_pred_ave_s"), ("bot", "tau_pred_bot_s")):
        df[f"e_{tag}_nodisc"] = (df.tau_fit_nodisc_s - df[pred]) / df[pred]
    fig_dir.mkdir(exist_ok=True, parents=True)
    return dict(df=df, meta=meta, fig=fig_dir,
                flagged=(df.fit_rmse_norm > meta["rmse_flag_threshold"]).to_numpy(),
                disc_error=meta["dt_fraction_of_tau"] / 2,
                rmse_threshold=meta["rmse_flag_threshold"],
                other_threshold=meta["other_group_threshold"])


VALIDITY_STUDIES = {
    "analytical_validity": load_validity(AV_DIR, "analytical_validity_data.csv", FIG_DIR),
    "analytical_validity2": load_validity(AV2_DIR, "analytical_validity2_data.csv", AV2_DIR / "fig"),
}

# thresholds are the same in both studies; keep module-level names for the helpers below
OTHER_THRESHOLD = VALIDITY_STUDIES["analytical_validity"]["other_threshold"]
RMSE_THRESHOLD = VALIDITY_STUDIES["analytical_validity"]["rmse_threshold"]

for _name, _s in VALIDITY_STUDIES.items():
    print(f"{_name}: {len(_s['df'])} samples, {int(_s['flagged'].sum())} non-exponential, "
          f"discretisation floor {_s['disc_error']:.1e} -> {_s['fig']}")

ERROR_SETS = {
    "": dict(tau_fit="tau_fitted_s", e_pred="e_pred", e_ave="e_ave", e_bot="e_bot",
             tau_lab=r"\tau_{\mathrm{fitted}}", disc=True),
    "_nodisc": dict(tau_fit="tau_fit_nodisc_s", e_pred="e_pred_nodisc", e_ave="e_ave_nodisc",
                    e_bot="e_bot_nodisc", tau_lab=r"\tau_{\mathrm{fitted}} - \Delta t/2",
                    disc=False),
}

FILL_ALPHA = {True: 0.85, False: 0.15}


def scatter_samples(ax, x, y, other1, other2, flagged, color=BASE_COLOR):
    """Two independent encodings, so neither hides the other:
      fill opacity -- opaque when the two governing groups that are NOT on the x axis are both
                      < OTHER_THRESHOLD, i.e. the trend against x is not polluted by them;
      edge colour  -- red when the decay is not exponential (fit RMSE > threshold), so that
                      tau_fitted is meaningless. The edge is always drawn opaque, including on
                      faded points, which is why the alpha goes in the face colour and not in
                      the `alpha` argument (that one would fade the edge too)."""
    from matplotlib.colors import to_rgba

    x, y = np.asarray(x, float), np.asarray(y, float)
    other1, other2 = np.asarray(other1, float), np.asarray(other2, float)
    flagged = np.asarray(flagged)
    clean = (other1 < OTHER_THRESHOLD) & (other2 < OTHER_THRESHOLD)
    for cl in (False, True):
        for fl in (False, True):
            m = (clean == cl) & (flagged == fl)
            if not m.any():
                continue
            ax.scatter(x[m], y[m], marker="o", s=22, linewidths=0.7,
                       facecolors=to_rgba(color, FILL_ALPHA[cl]),
                       edgecolors=FLAG_EDGE_COLOR if fl else BASE_EDGE_COLOR,
                       zorder=2 + cl + 2 * fl)


def _sci(v):
    """1e-05 -> '10^{-5}' for mathtext labels."""
    mant, exp = f"{v:e}".split("e")
    mant, exp = float(mant), int(exp)
    return rf"10^{{{exp}}}" if abs(mant - 1) < 1e-12 else rf"{mant:g}\times 10^{{{exp}}}"


def sample_handles(others_label, color=BASE_COLOR):
    """Proxy artists for the two independent encodings of scatter_samples: fill = whether the
    two groups not on the x axis are small (`others_label`), edge = whether the decay is
    exponential. The last handle keeps a faded fill on purpose: only its edge carries meaning."""
    from matplotlib.colors import to_rgba
    from matplotlib.lines import Line2D
    mk = dict(marker="o", linestyle="none", markersize=4.5)
    return [
        Line2D([0], [0], **mk, markerfacecolor=to_rgba(color, FILL_ALPHA[True]),
               markeredgecolor=BASE_EDGE_COLOR, label=others_label),
        Line2D([0], [0], **mk, markerfacecolor=to_rgba(color, FILL_ALPHA[False]),
               markeredgecolor=BASE_EDGE_COLOR, label="otherwise"),
        Line2D([0], [0], **mk, markerfacecolor=to_rgba(color, FILL_ALPHA[False]),
               markeredgecolor=FLAG_EDGE_COLOR,
               label=rf"$\mathrm{{RMSE}}_\mathrm{{fit}} > {_sci(RMSE_THRESHOLD)}$"),
    ]


# condition spelled out on each figure: the two groups that are not on the x axis
OTHERS = {
    "Pi": rf"$G_\mathrm{{mix}} < {OTHER_THRESHOLD:g}$, $G_P < {OTHER_THRESHOLD:g}$",
    "G_mix": rf"$\Pi < {OTHER_THRESHOLD:g}$, $G_P < {OTHER_THRESHOLD:g}$",
    "G_P": rf"$\Pi < {OTHER_THRESHOLD:g}$, $G_\mathrm{{mix}} < {OTHER_THRESHOLD:g}$",
}


def add_error_guides(ax, disc=None, signed=True, label_x=0.98):
    """Horizontal guides: y=0, +-5% (dotted) and, when `disc` is a value, the labelled
    discretisation floor dt/(2 tau) at +-disc."""
    if signed:
        ax.axhline(0, color="0.6", lw=0.6)
    for s in ((1, -1) if signed else (1,)):
        ax.axhline(s * 0.05, color="0.5", lw=0.7, ls=":")
        if disc:
            ax.axhline(s * disc, color="0.5", lw=0.7, ls="--")
    if disc:
        ax.text(label_x, disc, "discretization error", transform=ax.get_yaxis_transform(),
                fontsize=5.5, color="0.35", ha="right" if label_x > 0.5 else "left", va="center",
                bbox=dict(facecolor=plt.rcParams["axes.facecolor"], edgecolor="none", pad=0.5))

In [ ]:
# %% ===================== CELL A1 : parity plot =====================
# fitted decay time vs the analytical tau; y = x with a +-10% band, one dot per sample.
for study, S in VALIDITY_STUDIES.items():
    AV, AV_FLAGGED = S["df"], S["flagged"]
    for tag, s in ERROR_SETS.items():
        fig, ax = plt.subplots(figsize=(3.5, 2.8))

        x = AV.tau_pred_s.to_numpy() * SEC_TO_HOURS
        y = AV[s["tau_fit"]].to_numpy() * SEC_TO_HOURS
        from matplotlib.colors import to_rgba
        for fl in (False, True):                       # edge stays opaque on faded fills
            m = AV_FLAGGED == fl
            if m.any():
                ax.scatter(x[m], y[m], s=16, linewidths=0.6, zorder=3 + fl,
                           facecolors=to_rgba(BASE_COLOR, 0.7),
                           edgecolors=FLAG_EDGE_COLOR if fl else BASE_EDGE_COLOR)

        lims = [min(x.min(), y.min()) * 0.7, max(x.max(), y.max()) * 1.4]
        ref = np.array(lims)
        ax.plot(ref, ref, color="0.3", lw=1.0, zorder=1, label=r"$y=x$")
        ax.plot(ref, 1.1 * ref, color="0.3", lw=0.7, ls="--", zorder=1, label=r"$\pm 10\%$")
        ax.plot(ref, 0.9 * ref, color="0.3", lw=0.7, ls="--", zorder=1)

        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlim(lims)
        ax.set_ylim(lims)
        ax.set_xlabel(r"$\tau\ \mathrm{[h]}$ (analytical)")
        ax.set_ylabel(rf"${s['tau_lab']}\ \mathrm{{[h]}}$ (1D ARD)")
        ax.legend(loc="upper left", fontsize=7)

        frac = float((np.abs(y - x) / x <= 0.10).mean())
        ax.text(0.97, 0.05, f"{frac:.0%} within " + r"$\pm10\%$", transform=ax.transAxes,
                fontsize=7, ha="right", color="0.2")

        fig.tight_layout()
        fig.savefig(S["fig"] / f"validity_parity{tag}.pdf")
        print(f"{study} parity{tag or ' (raw)'}: {frac:.1%} of samples within +-10% (n={len(AV)})")

In [ ]:
# %% ===================== CELL A2 : |error| vs Pi, SPP vs corrected tau =====================
# absolute error of the SPP extraction time (grey) and of the corrected one (blue), which
# includes the Pi/(1-exp(-Pi)) saturation factor.
for study, S in VALIDITY_STUDIES.items():
    AV, AV_FLAGGED = S["df"], S["flagged"]
    for tag, s in ERROR_SETS.items():
        fig, ax = plt.subplots(figsize=(3.5, 2.8))

        for col, color in [(s["e_ave"], SPP_COLOR), (s["e_pred"], BASE_COLOR)]:
            scatter_samples(ax, AV.Pi, AV[col].abs(), AV.G_mix, AV.G_P, AV_FLAGGED, color=color)

        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel(r"$\langle\Pi\rangle$")
        ax.set_ylabel(rf"$|e| = |{s['tau_lab']}-\tau|\,/\,\tau$")
        add_error_guides(ax, disc=S["disc_error"] if s["disc"] else None, signed=False)
        ax.text(0.02, 0.05, r"$5\%$", transform=ax.get_yaxis_transform(), fontsize=5.5,
                color="0.35", ha="left", va="bottom")

        series = ax.legend(handles=sample_handles(r"$\tau_{\mathrm{SPP}}$", SPP_COLOR)[:1]
                           + sample_handles(r"$\tau = \tau_{\mathrm{SPP}}\,\Pi/(1-e^{-\Pi})$")[:1],
                           loc="upper left", fontsize=6, handlelength=1.2, labelspacing=0.3)
        ax.add_artist(series)
        ax.legend(handles=sample_handles(OTHERS["Pi"]), loc="lower right", fontsize=5.5,
                  handlelength=1.2, labelspacing=0.3)

        fig.tight_layout()
        fig.savefig(S["fig"] / f"validity_abs_error_vs_Pi{tag}.pdf")
        hi = AV.Pi > 1
        for name, col in [("tau_SPP", s["e_ave"]), ("tau corrected", s["e_pred"])]:
            print(f"{study:21s} {tag or '(raw)':9s} {name:14s}: median |e| = {AV[col].abs().median():.4f} "
                  f"(Pi>1: {AV[col][hi].abs().median():.3f}), "
                  f"{(AV[col].abs() <= 0.10).mean():.0%} within +-10%")

In [ ]:
# %% ===================== CELL A3 : signed error vs Pi =====================
for study, S in VALIDITY_STUDIES.items():
    AV, AV_FLAGGED = S["df"], S["flagged"]
    for tag, s in ERROR_SETS.items():
        fig, ax = plt.subplots(figsize=(3.5, 2.8))

        scatter_samples(ax, AV.Pi, AV[s["e_pred"]], AV.G_mix, AV.G_P, AV_FLAGGED)

        ax.set_xscale("log")
        ax.set_yscale("symlog", linthresh=0.01)
        ax.set_xlabel(r"$\langle\Pi\rangle$")
        ax.set_ylabel(rf"$e = \dfrac{{{s['tau_lab']}-\tau}}{{\tau}}$")
        add_error_guides(ax, disc=S["disc_error"] if s["disc"] else None)
        ax.legend(handles=sample_handles(OTHERS["Pi"]), loc="lower left", fontsize=5.5,
                  handlelength=1.2, labelspacing=0.3)

        fig.tight_layout()
        fig.savefig(S["fig"] / f"validity_error_vs_Pi{tag}.pdf")

In [ ]:
# %% ===================== CELL A4 : error vs G_mix =====================
for study, S in VALIDITY_STUDIES.items():
    AV, AV_FLAGGED = S["df"], S["flagged"]
    for tag, s in ERROR_SETS.items():
        fig, ax = plt.subplots(figsize=(3.5, 2.8))

        scatter_samples(ax, AV.G_mix, AV[s["e_pred"]], AV.Pi, AV.G_P, AV_FLAGGED)

        ax.set_xscale("log")
        ax.set_yscale("symlog", linthresh=0.01)
        ax.set_xlabel(r"$G_{\mathrm{mix}} = (H^2/E_l)\,/\,\tau_{\mathrm{fitted}}$")
        ax.set_ylabel(rf"$e = \dfrac{{{s['tau_lab']}-\tau}}{{\tau}}$")
        add_error_guides(ax, disc=S["disc_error"] if s["disc"] else None)
        ax.legend(handles=sample_handles(OTHERS["G_mix"]), loc="lower left", fontsize=5.5,
                  handlelength=1.2, labelspacing=0.3)

        fig.tight_layout()
        fig.savefig(S["fig"] / f"validity_error_vs_Gmix{tag}.pdf")

In [ ]:
# %% ===================== CELL A5 : error vs G_P =====================
# left = bottom-evaluated tau_0, right = the corrected height-averaged tau
for study, S in VALIDITY_STUDIES.items():
    AV, AV_FLAGGED = S["df"], S["flagged"]
    for tag, s in ERROR_SETS.items():
        fig, (axb, axa) = plt.subplots(1, 2, figsize=(7.0, 3.0), sharex=True, sharey=True)

        scatter_samples(axb, AV.G_P, AV[s["e_bot"]], AV.Pi, AV.G_mix, AV_FLAGGED)
        scatter_samples(axa, AV.G_P, AV[s["e_pred"]], AV.Pi, AV.G_mix, AV_FLAGGED)

        axb.set_ylabel(rf"$e_0 = \dfrac{{{s['tau_lab']}-\tau_0}}{{\tau_0}}$")
        axa.set_ylabel(rf"$e = \dfrac{{{s['tau_lab']}-\tau}}{{\tau}}$")
        for ax, title, lab in [(axb, r"bottom-evaluated $\tau_0$", "(a)"),
                               (axa, r"height-averaged $\tau$", "(b)")]:
            ax.set_xscale("log")
            ax.set_yscale("symlog", linthresh=0.01)
            ax.set_xlabel(r"$G_P$")
            ax.set_title(title, fontsize=9)
            ax.text(0.03, 0.94, lab, transform=ax.transAxes, fontweight="bold", va="top")
            add_error_guides(ax, disc=S["disc_error"] if s["disc"] else None)
        axa.legend(handles=sample_handles(OTHERS["G_P"]), loc="lower left", fontsize=5.5,
                   handlelength=1.2, labelspacing=0.3)

        fig.tight_layout()
        fig.savefig(S["fig"] / f"validity_error_vs_GP{tag}.pdf")

In [ ]:
# %% ===================== CELL A6 : text summary =====================
# for each design: how it sampled the space, then (i) the effect of each group alone, on the
# population where the OTHER two are small, and (ii) the joint effect of Pi and G_P, which is
# where the analytical tau actually breaks down.
PI_BINS = [(0, 0.1), (0.1, 1), (1, 10), (10, np.inf)]
GP_BINS = [(0, 0.1), (0.1, 1), (1, np.inf)]

for study, S in VALIDITY_STUDIES.items():
    AV, AV_FLAGGED = S["df"], S["flagged"]
    ok = ~AV_FLAGGED
    groups = {"Pi": AV.Pi.to_numpy(), "G_mix": AV.G_mix.to_numpy(), "G_P": AV.G_P.to_numpy()}

    print(f"\n{'=' * 78}\n{study}: {len(AV)} samples, {int(AV_FLAGGED.sum())} non-exponential, "
          f"discretisation floor {S['disc_error']:.1e}\n{'=' * 78}")
    print("sampling: " + S["meta"].get("description", "see metadata.json"))
    print("--- space covered, and how correlated the groups came out ---")
    for name, g in groups.items():
        print(f"{name:6s}: {g.min():.2e} .. {g.max():.2e}")
    lg = {k: np.log10(v) for k, v in groups.items()}
    for a, b in (("Pi", "G_P"), ("Pi", "G_mix"), ("G_P", "G_mix")):
        print(f"   corr(log {a}, log {b}) = {np.corrcoef(lg[a], lg[b])[0, 1]:+.3f}")
    if "eps_gH" in AV:
        limit = S["meta"].get("no_coalescence_eps_g_limit", 0.01)
        print(f"eps_g(H): {AV.eps_gH.min():.2e} .. {AV.eps_gH.max():.2e}; "
              f"{(AV.eps_gH > limit).sum()}/{len(AV)} above the {limit:.0%} no-coalescence limit")

    for tag, s in ERROR_SETS.items():
        e = AV[s["e_pred"]].to_numpy()
        print(f"\n--- {'raw tau_fitted' if not tag else 'tau_fitted - dt/2'} ---")
        print(f"overall: median |e| = {np.median(np.abs(e)):.2e}, "
              f"{(np.abs(e) <= 0.05).mean():.0%} within +-5%, {(np.abs(e) <= 0.10).mean():.0%} within +-10%")
        print(f"SPP tau: median |e| = {AV[s['e_ave']].abs().median():.2e}, "
              f"{(AV[s['e_ave']].abs() <= 0.10).mean():.0%} within +-10%")
        print(f"signed error range: {e.min():+.3f} .. {e.max():+.3f}")

        print(f"each group alone (the other two < {S['other_threshold']:.2g}, non-flagged):")
        for name, g in groups.items():
            others = [v for k, v in groups.items() if k != name]
            clean = (others[0] < S["other_threshold"]) & (others[1] < S["other_threshold"]) & ok
            if not clean.any():
                continue
            print(f"   {name:6s}: n={int(clean.sum()):3d}, spans {g[clean].min():.1e}..{g[clean].max():.1e}, "
                  f"median |e| = {np.median(np.abs(e[clean])):.2e}, max |e| = {np.max(np.abs(e[clean])):.2e}")

        print("median signed error on a Pi x G_P grid (non-flagged, n in parentheses):")
        print(f"{'Pi \\ G_P':>12s}" + "".join(f"{f'{a:g}-{b:g}':>16s}" for a, b in GP_BINS))
        for pa, pb in PI_BINS:
            row = f"{f'{pa:g}-{pb:g}':>12s}"
            for ga, gb in GP_BINS:
                m = (ok & (groups["Pi"] >= pa) & (groups["Pi"] < pb)
                     & (groups["G_P"] >= ga) & (groups["G_P"] < gb))
                row += f"{(f'{np.median(e[m]):+.3f} ({m.sum()})' if m.any() else '-'):>16s}"
            print(row)

In [ ]:
# %% ===================== CELL X0 : load the factorial study =====================
""" Factorial (G_P x Pi x G_mix) grid: each group has its own exact knob, so the three
    can be varied independently -- which the log-uniform sampling could not do.
    Figures of this study stay in its own run folder while we iterate. """
FACT_DIR = Path("runs/factorial_study")
FACT_FIG = FACT_DIR / "fig"
FACT_FIG.mkdir(exist_ok=True, parents=True)

_f_meta = json.load(open(FACT_DIR / "metadata.json"))
FA = pd.read_csv(FACT_DIR / "factorial_data.csv")
FA["e"] = (FA.tau_fitted_s - FA.dt_s / 2 - FA.tau_pred_s) / FA.tau_pred_s   # dt/2 bias removed
FA_FLAGGED = (FA.fit_rmse_norm > _f_meta["rmse_flag_threshold"]).to_numpy()

PI_LEVELS = _f_meta["Pi_levels"]
GP_LEVELS = _f_meta["G_P_levels"]
print(f"{len(FA)} runs in {_f_meta['elapsed_s']:.0f} s; "
      f"{int(FA_FLAGGED.sum())} non-exponential; n_cells {FA.n_cells.min()}..{FA.n_cells.max()}")
print("G_mix reachable at each G_P level (the correlation the sampled study could not break):")
for gp in GP_LEVELS:
    g = FA.G_mix_pred[np.isclose(FA.G_P_target, gp)]
    print(f"   G_P={gp:<5g}: G_mix {g.min():.2e} .. {g.max():.2e}")

In [ ]:
# %% ===================== CELL X1 : |error| vs G_mix, one panel per (Pi, G_P) =====================
# the whole grid at once: down = Pi, across = G_P, within a panel = G_mix (via E_l)
fig, axes = plt.subplots(len(PI_LEVELS), len(GP_LEVELS), figsize=(7.0, 6.4),
                         sharex=True, sharey=True)

for i, pi in enumerate(PI_LEVELS):
    for j, gp in enumerate(GP_LEVELS):
        ax = axes[i, j]
        m = np.isclose(FA.Pi_target, pi) & np.isclose(FA.G_P_target, gp)
        sub = FA[m].assign(flag=FA_FLAGGED[m]).sort_values("G_mix_pred")
        ax.plot(sub.G_mix_pred, sub.e.abs(), "-", color="0.6", lw=0.8, zorder=1)
        ax.scatter(sub.G_mix_pred, sub.e.abs(), s=18, zorder=3,
                   facecolors=BASE_COLOR, linewidths=0.7,
                   edgecolors=list(np.where(sub.flag, FLAG_EDGE_COLOR, BASE_EDGE_COLOR)))
        ax.axhline(0.05, color="0.5", lw=0.6, ls=":")
        ax.set_xscale("log")
        ax.set_yscale("log")
        if i == 0:
            ax.set_title(rf"$G_P = {gp:g}$", fontsize=8)
        if j == len(GP_LEVELS) - 1:
            ax.text(1.04, 0.5, rf"$\Pi = {pi:g}$", transform=ax.transAxes, fontsize=8,
                    rotation=270, va="center")

fig.supxlabel(r"$G_{\mathrm{mix}}$", fontsize=9)
fig.supylabel(r"$|e|$  (analytical $\tau$ vs 1D ARD, $\Delta t/2$ removed)", fontsize=9)
fig.tight_layout()
fig.savefig(FACT_FIG / "factorial_grid.pdf")

In [ ]:
# %% ===================== CELL X2 : one group at a time =====================
# each curve varies one group with the other two pinned at their smallest level, which the
# factorial design makes exact -- no "clean subset" to carve out afterwards
lo_pi, lo_gp, lo_el = min(PI_LEVELS), min(GP_LEVELS), max(_f_meta["E_l_scales"])
CURVES = {
    r"$\langle\Pi\rangle$": (np.isclose(FA.G_P_target, lo_gp) & np.isclose(FA.e_l_scale, lo_el),
                             "Pi", "#4477AA", "o"),
    r"$G_P$": (np.isclose(FA.Pi_target, lo_pi) & np.isclose(FA.e_l_scale, lo_el),
               "G_P", "#EE7733", "s"),
    r"$G_{\mathrm{mix}}$": (np.isclose(FA.Pi_target, lo_pi) & np.isclose(FA.G_P_target, lo_gp),
                            "G_mix_pred", "#228833", "^"),
}

fig, ax = plt.subplots(figsize=(3.5, 2.8))
for label, (mask, col, color, mark) in CURVES.items():
    sub = FA[mask].sort_values(col)
    ax.plot(sub[col], sub.e.abs(), marker=mark, color=color, ms=4, mfc="white", label=label)
    print(f"{label:22s} varied alone: " + ", ".join(
        f"{v:.3g} -> {abs(e):.1e}" for v, e in zip(sub[col], sub.e)))

ax.set_xscale("log")
ax.set_yscale("log")
ax.axhline(0.05, color="0.5", lw=0.7, ls=":")
ax.text(0.02, 0.05, r"$5\%$", transform=ax.get_yaxis_transform(), fontsize=6,
        color="0.35", ha="left", va="bottom")
ax.set_xlabel("value of the group being varied")
ax.set_ylabel(r"$|e|$")
ax.legend(loc="upper left", fontsize=7, title="varied alone", title_fontsize=7)
fig.tight_layout()
fig.savefig(FACT_FIG / "factorial_one_at_a_time.pdf")

In [ ]:
# %% ===================== CELL X3 : loss of the exponential decay =====================
# the second failure mode: past some G_mix the inventory no longer decays as a single
# exponential, so tau stops being the right descriptor. Shown at Pi = min so that the only
# ingredients are G_mix and G_P: poor mixing alone is harmless, it needs a z-dependent
# extraction rate (i.e. G_P) to produce a non-exponential decay.
fig, ax = plt.subplots(figsize=(3.5, 2.8))

pi_iso = min(PI_LEVELS)
for gp, mark in zip(GP_LEVELS, ("o", "s", "^", "D")):
    m = np.isclose(FA.G_P_target, gp) & np.isclose(FA.Pi_target, pi_iso)
    sub = FA[m].sort_values("G_mix_pred")
    ax.plot(sub.G_mix_pred, sub.fit_rmse_norm, marker=mark, ms=3.5, lw=0.8, mfc="white",
            label=rf"$G_P = {gp:g}$")

ax.axhline(_f_meta["rmse_flag_threshold"], color=FLAG_EDGE_COLOR, lw=0.8, ls="--")
ax.text(0.02, _f_meta["rmse_flag_threshold"] * 1.3, "non-exponential",
        transform=ax.get_yaxis_transform(), fontsize=6, color=FLAG_EDGE_COLOR, va="bottom")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$G_{\mathrm{mix}}$")
ax.set_ylabel(r"normalized RMSE of the exponential fit")
ax.legend(loc="upper left", fontsize=6, title=rf"$\Pi = {pi_iso:g}$", title_fontsize=6)
fig.tight_layout()
fig.savefig(FACT_FIG / "factorial_fit_rmse.pdf")

In [ ]:
# %% ===================== CELL R1 : SPP and PPL limiting regimes =====================
# tau_fitted / tau_SPP against <Pi>, with the analytical factor Pi/(1-exp(-Pi)) and its two
# asymptotes. Where the points sit on the lower asymptote the system is mass transfer limited
# (SPP), where they sit on the upper one it is partial pressure limited (PPL).
S = VALIDITY_STUDIES["analytical_validity2"]
AV, AV_FLAGGED = S["df"], S["flagged"]
tau_sim = AV.tau_fit_nodisc_s.to_numpy()
Pi = AV.Pi.to_numpy()

fig, ax = plt.subplots(figsize=(3.5, 2.8))
scatter_samples(ax, Pi, tau_sim / AV.tau_pred_ave_s, AV.G_mix, AV.G_P, AV_FLAGGED)

pi_line = np.logspace(np.log10(Pi.min()), np.log10(Pi.max()), 300)
ax.plot(pi_line, -pi_line / np.expm1(-pi_line), color="0.15", lw=1.1, zorder=5,
        label=r"$\Pi/(1-e^{-\Pi})$")
ax.axhline(1, color="#4477AA", lw=0.8, ls="--", zorder=4,
           label=r"SPP asymptote, $\tau \to \tau_{\mathrm{SPP}}$")
ax.plot(pi_line, pi_line, color="#EE7733", lw=0.8, ls="--", zorder=4,
        label=r"PPL asymptote, $\tau \to \tau_{\mathrm{PPL}}$")

ax.set_xscale("log")
ax.set_yscale("log")
ratio_all = tau_sim / AV.tau_pred_ave_s.to_numpy()
ax.set_ylim(0.7, ratio_all.max() * 2)          # the PPL asymptote runs off below; clip to the data
ax.set_xlabel(r"$\langle\Pi\rangle$")
ax.set_ylabel(r"$\tau_{\mathrm{fitted}} / \tau_{\mathrm{SPP}}$")
ax.legend(loc="upper left", fontsize=6)
fig.tight_layout()
fig.savefig(S["fig"] / "regimes_tau_vs_Pi.pdf")

# where does each asymptote hold to better than 5%? (clean samples only)
clean = (~AV_FLAGGED) & (AV.G_mix < S["other_threshold"]).to_numpy() & (AV.G_P < S["other_threshold"]).to_numpy()
for name, ratio in (("SPP", tau_sim / AV.tau_pred_ave_s.to_numpy()),
                    ("PPL", tau_sim / AV.tau_pred_PPL_s.to_numpy())):
    ok5 = clean & (np.abs(ratio - 1) < 0.05)
    if ok5.any():
        lo, hi = Pi[ok5].min(), Pi[ok5].max()
        print(f"{name} holds within 5% for <Pi> in {lo:.3g} .. {hi:.3g}  (n={int(ok5.sum())})")
print(f"tau_PPL / tau_PPL_outlet spans {(AV.tau_pred_PPL_s / AV.tau_pred_PPL_outlet_s).min():.2f}"
      f" .. {(AV.tau_pred_PPL_s / AV.tau_pred_PPL_outlet_s).max():.2f} (the two PPL forms differ by G_P alone)")

In [ ]:
# %% ===================== CELL D1 : design space, effect of the top pressure =====================
# At fixed molar gas flow, lowering P_top expands the bubbles: eps_g and the interfacial area
# grow, so the extraction gets faster. Restricted to the SPP samples, where tau does not depend
# on K_s, so the trend is the hydrodynamic effect alone. Shaded: where eps_g(H) exceeds the 1%
# void fraction behind the "no bubble coalescence" assumption, i.e. where the model is
# extrapolating beyond its own premises.
S = VALIDITY_STUDIES["analytical_validity2"]
AV, AV_FLAGGED = S["df"], S["flagged"]
EPS_LIMIT = S["meta"].get("no_coalescence_eps_g_limit", 0.01)

spp = (AV.Pi < 0.1).to_numpy() & ~AV_FLAGGED
p_atm = (AV.P_top_Pa / 101325).to_numpy()
tau_h = AV.tau_fit_nodisc_s.to_numpy() * SEC_TO_HOURS

fig, ax = plt.subplots(figsize=(3.5, 2.8))
bad = (AV.eps_gH > EPS_LIMIT).to_numpy()
if bad.any():
    ax.axvspan(p_atm.min() * 0.7, p_atm[bad].max(), color=FLAG_EDGE_COLOR, alpha=0.08, zorder=0)
    ax.text(p_atm[bad].max(), 0.02, r"$\varepsilon_g(H) > 1\%$, coalescence likely ",
            transform=ax.get_xaxis_transform(), fontsize=5.5, color=FLAG_EDGE_COLOR,
            va="bottom", ha="right", rotation=90)

ax.scatter(p_atm[~spp], tau_h[~spp], s=14, facecolors="0.75", edgecolors="none",
           alpha=0.5, zorder=2, label=r"$\langle\Pi\rangle \geq 0.1$")
ax.scatter(p_atm[spp], tau_h[spp], s=18, facecolors=BASE_COLOR, edgecolors=BASE_EDGE_COLOR,
           linewidths=0.6, alpha=0.85, zorder=3, label=r"$\langle\Pi\rangle < 0.1$ (SPP)")

slope, intercept = np.polyfit(np.log10(p_atm[spp]), np.log10(tau_h[spp]), 1)
pl = np.logspace(np.log10(p_atm.min()), np.log10(p_atm.max()), 50)
ax.plot(pl, 10**intercept * pl**slope, color="0.15", lw=1.0, ls=(0, (3, 2)), zorder=5,
        label=rf"$\tau \propto P_{{\mathrm{{top}}}}^{{{slope:.2f}}}$")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(p_atm.min() * 0.7, p_atm.max() * 1.4)
ax.set_xlabel(r"$P_{\mathrm{top}}\ \mathrm{[atm]}$")
ax.set_ylabel(r"$\tau_{\mathrm{fitted}}\ \mathrm{[h]}$")
ax.legend(loc="upper left", fontsize=6)
fig.tight_layout()
fig.savefig(S["fig"] / "design_tau_vs_Ptop.pdf")

print(f"SPP subset (n={int(spp.sum())}): tau ~ P_top^{slope:.3f}")
lo, hi = p_atm[spp].min(), p_atm[spp].max()
print(f"   over P_top {lo:.3g} .. {hi:.3g} atm, tau ranges {tau_h[spp].min():.3g} .. {tau_h[spp].max():.3g} h")
ok = spp & ~bad
if ok.any():
    s_ok = np.polyfit(np.log10(p_atm[ok]), np.log10(tau_h[ok]), 1)[0]
    print(f"   restricted to eps_g(H) < {EPS_LIMIT:.0%} (n={int(ok.sum())}): exponent {s_ok:.3f}")
print(f"   d_b(0) spans {AV.d_b0_m.min() * 1e3:.2f} .. {AV.d_b0_m.max() * 1e3:.2f} mm "
      f"(bubble diameter correlation is extrapolated at both ends of the P_top range)")

In [ ]:
# %% ===================== CELL N0 : load the non-exponential case =====================
""" One analytical_validity2 sample flagged as non-exponential while Pi and G_P are both
    small: the liquid is badly mixed (G_mix >> 1) but the extraction rate is nearly uniform
    along z, so tau keeps its value while the decay stops being a single exponential. """
NONEXP_STUDIES = {}
for _d in ("runs/non_exponential", "runs/non_exponential2"):
    _d = Path(_d)
    meta = json.load(open(_d / "metadata.json"))
    (_d / "fig").mkdir(exist_ok=True, parents=True)
    NONEXP_STUDIES[_d.name] = dict(
        meta=meta, rec=meta["record"], fig=_d / "fig", tau=meta["record"]["tau_pred_s"],
        exports={n: load_export(_d / f"{n}.csv") for n in ("c_T2", "P_T2", "aJ_T2", "n_T2_salt")},
    )

for _name, _s in NONEXP_STUDIES.items():
    r = _s["rec"]
    print(f"{_name} (sample {_s['meta']['sample_id']}): Pi={r['Pi']:.3g}, G_P={r['G_P']:.3g}, "
          f"G_mix={r['G_mix_pred']:.3g}")
    print(f"   tau_pred={r['tau_pred_s'] / HOURS_TO_SEC:.2f} h, "
          f"tau_fitted={r['tau_fitted_s'] / HOURS_TO_SEC:.2f} h "
          f"({r['tau_fitted_s'] / r['tau_pred_s'] - 1:+.1%}), fit RMSE={r['fit_rmse_norm']:.3e}")

In [ ]:
# %% ===================== CELL N1 : profiles when the liquid is badly mixed =====================
# liquid concentration, gas tritium pressure and volumetric transfer rate, shared z axis.
# With G_mix >> 1 the liquid can no longer flatten its own profile over one extraction time,
# so c_T2 keeps the shape the source term gives it instead of staying uniform.
for study, S in NONEXP_STUDIES.items():
    NE, NE_REC, NE_TAU = S["exports"], S["rec"], S["tau"]
    fig, axes = plt.subplots(3, 1, figsize=(3.5, 5.4), sharex=True)

    idx = select_nearest(NE["c_T2"]["times"], np.array([0.05, 0.5, 1.0, 2.0]) * NE_TAU)
    colors = plt.cm.viridis(np.linspace(0.05, 0.8, len(idx)))

    for ax, name, ylabel in zip(
        axes,
        ("c_T2", "P_T2", "aJ_T2"),
        (r"$c_{T_2}\ \mathrm{[mol\,m^{-3}]}$", r"$P_{T_2}\ \mathrm{[Pa]}$",
         r"$a\,J_{T_2}\ \mathrm{[mol\,m^{-3}\,s^{-1}]}$"),
    ):
        e = NE[name]
        for c, i in zip(colors, idx):
            ax.plot(e["x"] * M_TO_CM, e["data"][i], color=c, lw=1.3,
                    label=fr"$t = {e['times'][i] / NE_TAU:.2f}\,\tau$")
        ax.set_ylabel(ylabel, fontsize=8)
        ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 2))

    axes[0].legend(loc="upper right", fontsize=6)
    axes[-1].set_xlabel(r"$z\ \mathrm{[cm]}$")
    axes[-1].set_xlim(0, NE["c_T2"]["x"].max() * M_TO_CM)
    for ax, lab in zip(axes, ("(a)", "(b)", "(c)")):
        ax.text(0.02, 0.94, lab, transform=ax.transAxes, fontweight="bold", va="top", fontsize=8)

    fig.tight_layout()
    fig.savefig(S["fig"] / "nonexp_profiles.pdf")

    c0 = NE["c_T2"]["data"]
    i_tau = select_nearest(NE["c_T2"]["times"], [NE_TAU])[0]
    print(f"{study}: c_T2 non-uniformity (max-min)/mean: t=0.05 tau -> "
          f"{np.ptp(c0[idx[0]]) / c0[idx[0]].mean():.2f}, t=1 tau -> {np.ptp(c0[i_tau]) / c0[i_tau].mean():.2f}")

In [ ]:
# %% ===================== CELL N2 : is the exponential fit visibly off? =====================
# the fit that the study performs, overlaid on the inventory, plus the residual it leaves --
# the residual is the only place where the departure from a single exponential is visible.
from sparging.postprocess import fit_exp

for study, S in NONEXP_STUDIES.items():
    NE, NE_REC, NE_TAU = S["exports"], S["rec"], S["tau"]
    inv = NE["n_T2_salt"]
    t, n = inv["times"], inv["data"]
    (tau_fit, n0_fit), _ = fit_exp(n * ureg.molT2, t * ureg.s, t[0] * ureg.s, t[-1] * ureg.s,
                                   "decay", tau_guess=NE_TAU * ureg.s)
    tau_fit_s = tau_fit.to("s").magnitude
    n_fit = n0_fit.to("molT2").magnitude * np.exp(-t / tau_fit_s)

    fig, (ax, axr) = plt.subplots(2, 1, figsize=(3.5, 3.6), sharex=True,
                                  gridspec_kw={"height_ratios": [2.2, 1]})

    ax.plot(t / NE_TAU, n, color=BASE_COLOR, **SIM_KW, label="1D ARD model")
    ax.plot(t / NE_TAU, n_fit, **ANA_KW, label=rf"fit, $\tau = {tau_fit_s / HOURS_TO_SEC:.2f}$ h")
#     ax.set_yscale("log")
    ax.set_xlim(0, t.max() / NE_TAU)
    ax.set_ylabel(r"$n_{T_2}\ \mathrm{[mol]}$")
    ax.legend(loc="lower left", fontsize=7)

    axr.plot(t / NE_TAU, (n - n_fit) / n_fit * 100, color=BASE_COLOR, lw=1.1)
    axr.axhline(0, color="0.6", lw=0.6)
    axr.set_xlabel(r"$t/\tau$")
    axr.set_ylabel("residual [%]", fontsize=8)

    fig.tight_layout()
    fig.savefig(S["fig"] / "nonexp_inventory_fit.pdf")

    res = (n - n_fit) / n_fit
    print(f"{study}: exponential fit: tau = {tau_fit_s / HOURS_TO_SEC:.4f} h vs analytical "
          f"{NE_TAU / HOURS_TO_SEC:.4f} h ({tau_fit_s / NE_TAU - 1:+.3%})")
    print(f"residual: max |.| = {np.abs(res).max():.2%}, and it is systematic (sign changes "
          f"{int(np.sum(np.diff(np.sign(res[1:])) != 0))} times over the window)")
    thr = VALIDITY_STUDIES["analytical_validity2"]["rmse_threshold"]
    verdict = ("visible by eye on the decay itself" if np.abs(res).max() > 0.02
               else "far below what the eye can see on the decay itself")
    print(f"normalized RMSE = {NE_REC['fit_rmse_norm']:.3e} = {NE_REC['fit_rmse_norm'] / thr:.0f}x "
          f"the flag threshold -> departure {verdict}")